In [2]:
from dataclasses import dataclass


@dataclass
class TokenizerConfig:
    max_length: int = 32


class CharTokenizer:
    def __init__(self, config: TokenizerConfig):
        self.config = config

        characters = list(
            "abcdefghijklmnopqrstuvwxyz0123456789 ,.-"
        )

        special_tokens = [
            "<pad>",
            "<bos>",
            "<eos>",
            "<unk>",
        ]

        self.tokens = special_tokens + characters

        self.token_to_id = {
            token: index
            for index, token in enumerate(self.tokens)
        }

        self.id_to_token = {
            index: token
            for token, index in self.token_to_id.items()
        }

        self.pad_token_id = self.token_to_id["<pad>"]
        self.bos_token_id = self.token_to_id["<bos>"]
        self.eos_token_id = self.token_to_id["<eos>"]
        self.unk_token_id = self.token_to_id["<unk>"]

    @property
    def vocab_size(self) -> int:
        return len(self.tokens)

    def encode(self, text: str) -> list[int]:
        text = text.lower()

        token_ids = [self.bos_token_id]

        for character in text:
            token_ids.append(
                self.token_to_id.get(
                    character,
                    self.unk_token_id,
                )
            )

        token_ids.append(self.eos_token_id)

        token_ids = token_ids[: self.config.max_length]

        if token_ids[-1] != self.eos_token_id:
            token_ids[-1] = self.eos_token_id

        padding_length = (
            self.config.max_length - len(token_ids)
        )

        token_ids.extend(
            [self.pad_token_id] * padding_length
        )

        return token_ids

    def decode(self, token_ids: list[int]) -> str:
        characters = []

        for token_id in token_ids:
            if token_id == self.eos_token_id:
                break

            if token_id in {
                self.pad_token_id,
                self.bos_token_id,
            }:
                continue

            token = self.id_to_token.get(
                token_id,
                "",
            )

            if not token.startswith("<"):
                characters.append(token)

        return "".join(characters)

tokenizer = CharTokenizer(
    TokenizerConfig(max_length=32)
)

ids = tokenizer.encode("a red circle")

print(ids)
print(tokenizer.decode(ids))
print(tokenizer.vocab_size)

[1, 4, 40, 21, 8, 7, 40, 6, 12, 21, 6, 15, 8, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
a red circle
44


In [ ]:
import torch
import torch.nn as nn


class PatchEmbedding(nn.Module):
    def __init__(
        self,
        image_size: int = 64,
        patch_size: int = 8,
        in_channels: int = 3,
        embed_dim: int = 128,
    ):
        super().__init__()

        if image_size % patch_size != 0:
            raise ValueError(
                "image_size必须能被patch_size整除"
            )

        self.grid_size = image_size // patch_size
        self.num_patches = self.grid_size**2

        self.projection = nn.Conv2d(
            in_channels=in_channels,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(
        self,
        images: torch.Tensor,
    ) -> torch.Tensor:
        # [B, 3, 64, 64]
        features = self.projection(images)

        # [B, 128, 8, 8]
        features = features.flatten(2)

        # [B, 64, 128]
        features = features.transpose(1, 2)

        return features

In [ ]:
class TinyVisionEncoder(nn.Module):
    def __init__(
        self,
        image_size: int = 64,
        patch_size: int = 8,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 2,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.patch_embedding = PatchEmbedding(
            image_size=image_size,
            patch_size=patch_size,
            embed_dim=embed_dim,
        )

        num_patches = self.patch_embedding.num_patches

        self.position_embedding = nn.Parameter(
            torch.randn(
                1,
                num_patches,
                embed_dim,
            )
            * 0.02
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers,
        )

        self.norm = nn.LayerNorm(embed_dim)

    def forward(
        self,
        images: torch.Tensor,
    ) -> torch.Tensor:
        visual_tokens = self.patch_embedding(images)

        visual_tokens = (
            visual_tokens
            + self.position_embedding
        )

        visual_tokens = self.encoder(
            visual_tokens
        )

        return self.norm(visual_tokens)

In [ ]:
class VisualProjector(nn.Module):
    def __init__(
        self,
        vision_dim: int,
        language_dim: int,
    ):
        super().__init__()

        self.projector = nn.Sequential(
            nn.Linear(
                vision_dim,
                language_dim,
            ),
            nn.GELU(),
            nn.Linear(
                language_dim,
                language_dim,
            ),
        )

    def forward(
        self,
        visual_features: torch.Tensor,
    ) -> torch.Tensor:
        return self.projector(visual_features)

In [ ]:
class TinyCausalLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int = 128,
        num_heads: int = 4,
        num_layers: int = 4,
        max_sequence_length: int = 128,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.embed_dim = embed_dim

        self.token_embedding = nn.Embedding(
            vocab_size,
            embed_dim,
        )

        self.position_embedding = nn.Embedding(
            max_sequence_length,
            embed_dim,
        )

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
            activation="gelu",
        )

        self.transformer = nn.TransformerEncoder(
            decoder_layer,
            num_layers=num_layers,
        )

        self.norm = nn.LayerNorm(embed_dim)

        self.lm_head = nn.Linear(
            embed_dim,
            vocab_size,
            bias=False,
        )

        # 权重共享
        self.lm_head.weight = self.token_embedding.weight

    def create_causal_mask(
        self,
        sequence_length: int,
        device: torch.device,
    ) -> torch.Tensor:
        mask = torch.triu(
            torch.ones(
                sequence_length,
                sequence_length,
                device=device,
                dtype=torch.bool,
            ),
            diagonal=1,
        )

        return mask

    def forward_embeddings(
        self,
        embeddings: torch.Tensor,
    ) -> torch.Tensor:
        batch_size, sequence_length, _ = (
            embeddings.shape
        )

        positions = torch.arange(
            sequence_length,
            device=embeddings.device,
        )

        embeddings = (
            embeddings
            + self.position_embedding(positions)[None]
        )

        causal_mask = self.create_causal_mask(
            sequence_length,
            embeddings.device,
        )

        hidden_states = self.transformer(
            embeddings,
            mask=causal_mask,
        )

        hidden_states = self.norm(hidden_states)

        return self.lm_head(hidden_states)